# UTM Reference Sample, by Campaign Category

## What this notebook produces

A CSV file (`utm_sample.csv`) of real, representative UTM/campaign name
examples for each of the ten campaign categories used throughout this
project's reactivation analyses -- intended as a quick reference for the
strategy and marketing teams to see what an actual send looks like under
each category label, not just the category name in the abstract.

For each category, this notebook selects up to 20 of its most-sent distinct
campaign/UTM combinations (by total send volume over the analysis window),
along with how many times each was sent and the date range it was active.
Three categories have fewer than 20 distinct combinations available in the
source data at all -- BIRTHDAY (17), CROSSSELL (1), and TRACKING_ARTIFACT
(1) -- so their sample includes everything that exists rather than a full
20; this is a fact about how narrow those categories are, not a gap in this
notebook's query.

## Campaign categories and how they're identified

Campaigns are classified from the sent campaign's name (`send_utm_campaign`
on the Blueshift send data), using pattern matching rather than a verified
per-send flag, since no such flag exists in the data available. A send is
assigned to the first category it matches, checked in this order:

| Order | Category | What it captures |
|---|---|---|
| 1 | **APEC** | Algorithmic Personalized Email Content -- names containing "apec", plus its confirmed browse-abandon win-back triggers, StylePass onboarding messages, and style-profile reactivation pushes. |
| 2 | **TRANSACTIONAL** | Account-service email (password resets, order/Fix status confirmations) -- a client only receives one because they're already taking an action on their account, not because of a marketing choice. |
| 3 | **DSE** | Dynamic Shoppable Email, including its TFY and CYL content variants. |
| 4 | **PROMO_INCENTIVE** (partial) | A one-time mass reactivation offer test (control/test arms) -- its name happens to also contain FLS's naming convention, so it's checked and routed here before the FLS check below, ahead of being misread as a behavioral FLS trigger. |
| 5 | **FLS** | Freestyle Lifecycle Series -- behaviorally triggered sends after a high-intent event (cart abandonment, a saved item still on sale), identified by an explicit "freestyle-lifecycle" naming convention. |
| 6 | **TRACKING_ARTIFACT** | The single `crumbs_has_app` tag -- a uniform internal event marker, not a real send with real creative content. |
| 7 | **PROMO_INCENTIVE** (remainder) | Campaign names referencing an offer, discount, sale, percent-off, markdown, deal, coupon, a "spend X get Y" mechanic, a last-chance framing, a credit, an incentive, or a waived styling fee. |
| 8 | **WINBACK** | Campaign names explicitly branded as reactivation or win-back messaging, with no financial incentive attached. |
| 9 | **BIRTHDAY** | Birthday-triggered sends. |
| 10 | **CROSSSELL** | Post-Fix cross-sell sends. |
| 11 | **OTHER** | Everything else -- generic seasonal, thematic, and welcome/onboarding marketing content with no personalization-, promotion-, or lifecycle-event-suggestive naming. |

## Scope

- **Send history window:** 2025-01-01 to 2026-05-31 (the same ~17-month
  window used throughout this project's analyses), restricted to sends
  that were actually delivered (`holdout_group = 0`), not just targeted.
- **Distinct combination** here means a unique (campaign name, UTM content,
  UTM source, trigger type) tuple -- the same campaign name can appear more
  than once in the sample if it was sent with different content variants.
- Selection is by **total send volume**, highest first -- this surfaces the
  most commonly-used, representative naming for each category rather than
  rare one-off test variants.

## Output columns

| Column | Meaning |
|---|---|
| Campaign Category | The derived category (APEC, DSE, FLS, etc.) |
| Blueshift Campaign | The raw `send_utm_campaign` value |
| UTM Content | The raw `send_utm_content` value |
| UTM Source | The raw `send_utm_source` value |
| Full UTM String | `send_utm_source` + `send_utm_campaign` + `send_utm_content` assembled into a standard query-string shape, for a quick copy-paste reference |
| Trigger Type | The Blueshift trigger mechanism (e.g. `EmailTrigger`, `PushTrigger`) |
| Number Of Sends | Total delivered sends for this exact combination, over the full window |
| First Sent Date | Earliest date this combination was sent in the window |
| Last Sent Date | Most recent date this combination was sent in the window |

**Running this notebook regenerates `utm_sample.csv` in this project
folder** as a separate, standalone output file -- it is not embedded in
the notebook itself.

In [1]:
import time

import pandas as pd
from amphibian import get_data_accessor

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)

def query(sql, max_attempts=4, retry_delay_seconds=20):
    # The longer queries in this project occasionally hit a transient connection
    # drop (network/VPN blip, not a query bug -- the same query succeeds on retry).
    for attempt in range(1, max_attempts + 1):
        try:
            return get_data_accessor(engines=['presto']).fetch_sql(sql=sql, error_on_empty=False)
        except Exception as e:
            if attempt == max_attempts:
                raise
            print(f"query() attempt {attempt}/{max_attempts} failed ({type(e).__name__}: {e}); retrying in {retry_delay_seconds}s...")
            time.sleep(retry_delay_seconds)

START_DATE = '2025-01-01'
END_DATE = '2026-05-31'
SAMPLES_PER_CATEGORY = 20
OUTPUT_CSV = 'utm_sample.csv'

print(f"Analysis window: {START_DATE} to {END_DATE}")

Analysis window: 2025-01-01 to 2026-05-31


## Step 1 — Pull every distinct UTM combination, with volume and date range

One row per distinct (category, campaign, content, source, trigger type)
combination actually sent in the window, with how many times it was sent
and the first/last date it appeared.

In [2]:
utm_combinations_query = f"""--sql
WITH sends AS (
    SELECT send_utm_campaign, send_utm_content, send_utm_source, trigger_type, sent_timestamp
    FROM blueshift.campaign_activity_kpis
    WHERE execution_date >= DATE '{START_DATE}'
      AND execution_date <= DATE '{END_DATE}'
      AND holdout_group = 0 -- only include sends that were actually delivered, not just targeted
),
categorized AS (
    SELECT
        send_utm_campaign, send_utm_content, send_utm_source, trigger_type, sent_timestamp,
        CASE
            WHEN send_utm_campaign LIKE '%apec%' THEN 'APEC'
            WHEN send_utm_campaign LIKE '%browseabandon%' THEN 'APEC'
            WHEN send_utm_campaign LIKE '%stylepass%' THEN 'APEC'
            WHEN send_utm_campaign LIKE '%styleprofile%' THEN 'APEC'
            WHEN send_utm_campaign LIKE '%transactional%' THEN 'TRANSACTIONAL'
            WHEN regexp_like(send_utm_campaign, '(^|_)dse(_|$)') THEN 'DSE'
            WHEN send_utm_campaign LIKE '%tfy%' THEN 'DSE'
            WHEN send_utm_campaign LIKE '%cyl%' THEN 'DSE'
            WHEN send_utm_campaign LIKE '%offertest%' THEN 'PROMO_INCENTIVE' -- shares FLS naming; a one-time mass offer test, checked before the FLS rule below
            WHEN send_utm_campaign LIKE '%freestyle-lifecycle%' THEN 'FLS'
            WHEN send_utm_campaign LIKE '%freestyle_lifecycle%' THEN 'FLS'
            WHEN send_utm_campaign = 'crumbs_has_app' THEN 'TRACKING_ARTIFACT'
            WHEN send_utm_campaign LIKE '%offer%' OR send_utm_campaign LIKE '%discount%'
              OR send_utm_campaign LIKE '%promo%' OR send_utm_campaign LIKE '%sale%'
              OR send_utm_campaign LIKE '%percentoff%' OR send_utm_campaign LIKE '%markdown%'
              OR send_utm_campaign LIKE '%deal%' OR send_utm_campaign LIKE '%coupon%'
              OR send_utm_campaign LIKE '%spendget%' OR send_utm_campaign LIKE '%lastchance%'
              OR send_utm_campaign LIKE '%credit%' OR send_utm_campaign LIKE '%incentive%'
              OR send_utm_campaign LIKE '%waivedstylingfee%'
            THEN 'PROMO_INCENTIVE'
            WHEN send_utm_campaign LIKE '%reactivation%' OR send_utm_campaign LIKE '%winback%'
            THEN 'WINBACK'
            WHEN send_utm_campaign LIKE '%birthday%'
            THEN 'BIRTHDAY'
            WHEN send_utm_campaign LIKE '%crossell%' OR send_utm_campaign LIKE '%crosssell%'
            THEN 'CROSSSELL'
            ELSE 'OTHER'
        END AS campaign_category
    FROM sends
)
SELECT
    campaign_category,
    send_utm_campaign,
    send_utm_content,
    send_utm_source,
    trigger_type,
    COUNT(*) AS n_sends,
    MIN(date(sent_timestamp)) AS first_sent_date,
    MAX(date(sent_timestamp)) AS last_sent_date
FROM categorized
GROUP BY 1, 2, 3, 4, 5
ORDER BY campaign_category, n_sends DESC
"""

utm_combinations_df = query(utm_combinations_query)
utm_combinations_df.head()

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,campaign_category,send_utm_campaign,send_utm_content,send_utm_source,trigger_type,n_sends,first_sent_date,last_sent_date
0,APEC,email_us_w_apec_fix_021225_sup,email_us_w_apec_fix_c_021225_sup_1209099621045247,blueshift,EmailTrigger,1884145,2025-02-12,2025-02-12
1,APEC,email_us_w_apec_fix_021425_sup,email_us_w_apec_fix_d_021425_sup_1209099621045247,blueshift,EmailTrigger,1883496,2025-02-14,2025-02-14
2,APEC,email_us_w_apec_fix_020725_sup,email_us_w_apec_fix_d_020725_sup_1209099621045247,blueshift,EmailTrigger,1883316,2025-02-07,2025-02-07
3,APEC,email_us_w_apec_fix_013125_sup,email_us_w_apec_fix_d_013125_sup_1208792458875519,blueshift,EmailTrigger,1882205,2025-01-31,2025-01-31
4,APEC,email_us_w_apec_fix_022125_sup,email_us_w_apec_fix_d_022125_sup_1209099621045247,blueshift,EmailTrigger,1882162,2025-02-21,2025-02-21


## Step 2 — How many distinct combinations exist per category

A quick check before sampling: some categories are inherently narrow (a
single canonical campaign name), so their sample can't reach 20 no matter
how the data is sliced.

In [3]:
combo_counts = utm_combinations_df.groupby('campaign_category').size().rename('distinct_combinations').sort_values(ascending=False)
combo_counts

campaign_category
OTHER                3154
DSE                  1725
APEC                  758
PROMO_INCENTIVE       746
TRANSACTIONAL         211
FLS                    69
WINBACK                55
BIRTHDAY               17
CROSSSELL               1
TRACKING_ARTIFACT       1
Name: distinct_combinations, dtype: int64

## Step 3 — Sample up to 20 per category, by send volume

Each category's rows are already sorted by `n_sends` descending (from the
query above), so taking the first 20 rows per category surfaces the
highest-volume, most representative combinations. Categories with fewer
than 20 distinct combinations simply contribute everything they have.

In [4]:
utm_sample_df = (
    utm_combinations_df
    .groupby('campaign_category', group_keys=False)
    .apply(lambda g: g.head(SAMPLES_PER_CATEGORY))
    .reset_index(drop=True)
)

utm_sample_df['full_utm_string'] = (
    'utm_source=' + utm_sample_df['send_utm_source'].fillna('').astype(str)
    + '&utm_campaign=' + utm_sample_df['send_utm_campaign'].fillna('').astype(str)
    + '&utm_content=' + utm_sample_df['send_utm_content'].fillna('').astype(str)
)

utm_sample_df = utm_sample_df.rename(columns={
    'campaign_category': 'Campaign Category',
    'send_utm_campaign': 'Blueshift Campaign',
    'send_utm_content': 'UTM Content',
    'send_utm_source': 'UTM Source',
    'full_utm_string': 'Full UTM String',
    'trigger_type': 'Trigger Type',
    'n_sends': 'Number Of Sends',
    'first_sent_date': 'First Sent Date',
    'last_sent_date': 'Last Sent Date',
})[[
    'Campaign Category', 'Blueshift Campaign', 'UTM Content', 'UTM Source',
    'Full UTM String', 'Trigger Type', 'Number Of Sends', 'First Sent Date', 'Last Sent Date',
]]

print(f"Total sample rows: {len(utm_sample_df)}")
utm_sample_df.groupby('Campaign Category').size().rename('rows_included')

Total sample rows: 159


Campaign Category
APEC                 20
BIRTHDAY             17
CROSSSELL             1
DSE                  20
FLS                  20
OTHER                20
PROMO_INCENTIVE      20
TRACKING_ARTIFACT     1
TRANSACTIONAL        20
WINBACK              20
Name: rows_included, dtype: int64

## Step 4 — Write the CSV and preview a few rows

In [5]:
utm_sample_df.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {len(utm_sample_df)} rows to {OUTPUT_CSV}")
utm_sample_df.head(20)

Wrote 159 rows to utm_sample.csv


,Campaign Category,Blueshift Campaign,UTM Content,UTM Source,Full UTM String,Trigger Type,Number Of Sends,First Sent Date,Last Sent Date
0,APEC,email_us_w_apec_fix_021225_sup,email_us_w_apec_fix_c_021225_sup_1209099621045247,blueshift,utm_source=blueshift&utm_campaign=email_us_w_apec_fix_02...,EmailTrigger,1884145,2025-02-12,2025-02-12
1,APEC,email_us_w_apec_fix_021425_sup,email_us_w_apec_fix_d_021425_sup_1209099621045247,blueshift,utm_source=blueshift&utm_campaign=email_us_w_apec_fix_02...,EmailTrigger,1883496,2025-02-14,2025-02-14
2,APEC,email_us_w_apec_fix_020725_sup,email_us_w_apec_fix_d_020725_sup_1209099621045247,blueshift,utm_source=blueshift&utm_campaign=email_us_w_apec_fix_02...,EmailTrigger,1883316,2025-02-07,2025-02-07
3,APEC,email_us_w_apec_fix_013125_sup,email_us_w_apec_fix_d_013125_sup_1208792458875519,blueshift,utm_source=blueshift&utm_campaign=email_us_w_apec_fix_01...,EmailTrigger,1882205,2025-01-31,2025-01-31
4,APEC,email_us_w_apec_fix_022125_sup,email_us_w_apec_fix_d_022125_sup_1209099621045247,blueshift,utm_source=blueshift&utm_campaign=email_us_w_apec_fix_02...,EmailTrigger,1882162,2025-02-21,2025-02-21
5,APEC,email_us_w_apec_fix_020525_sup,email_us_w_apec_fix_c_020525_sup_1209099621045247,blueshift,utm_source=blueshift&utm_campaign=email_us_w_apec_fix_02...,EmailTrigger,1882102,2025-02-05,2025-02-05
6,APEC,email_us_w_apec_fix_061125_sup,email_us_w_apec_fix_3_061125_sup_1210171829973398,blueshift,utm_source=blueshift&utm_campaign=email_us_w_apec_fix_06...,EmailTrigger,1882081,2025-06-11,2025-06-11
7,APEC,email_us_w_apec_fix_060425_sup,email_us_w_apec_fix_3_060425_sup_1210171829973398,blueshift,utm_source=blueshift&utm_campaign=email_us_w_apec_fix_06...,EmailTrigger,1881796,2025-06-04,2025-06-04
8,APEC,email_us_w_apec_fix_032125_sup,email_us_w_apec_fix_d_032125_sup_1209314645692159,blueshift,utm_source=blueshift&utm_campaign=email_us_w_apec_fix_03...,EmailTrigger,1881790,2025-03-21,2025-03-21
9,APEC,email_us_w_apec_fix_020225_sup,email_us_w_apec_fix_a_020225_sup_1209099621045247,blueshift,utm_source=blueshift&utm_campaign=email_us_w_apec_fix_02...,EmailTrigger,1881691,2025-02-02,2025-02-02
